# 🎯 Анализатор рекламы на GigaChat

Этот notebook обучает модель на ваших данных и тестирует её.

---

## 🚀 Как использовать:
1. Нажми **Run All** (▶️ Run All)
2. Жди обучения (~5-10 минут первый раз, потом 2 секунды)
3. Смотри результаты тестов

---

## Структура:
1. **Блок 1**: Подключение к GigaChat
2. **Блок 2**: Обучение модели (один раз долго, потом быстро)
3. **Блок 3**: Автоматическое тестирование

---


## 📡 Блок 1: Подключение к GigaChat

Настраиваем соединение с API GigaChat.


In [1]:
#!/usr/bin/env python3
import os
import json
import pandas as pd
import pickle
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
from typing import List, Dict

# Загружаем .env файл
load_dotenv()

# Проверяем наличие API ключа
API_KEY = os.getenv("API_KEY")

if not API_KEY:
    print("❌ ОШИБКА: API_KEY не найден в .env файле!")
    print("\n📝 Создайте .env файл со следующим содержимым:")
    print("   API_KEY=ваш_api_ключ_от_cloud.ru")
    raise ValueError("API_KEY не найден!")
else:
    print("✅ API ключ загружен успешно")
    print(f"   API_KEY: {API_KEY[:20]}...")

# Создаем клиент GigaChat
url = "https://foundation-models.api.cloud.ru/v1"

client = OpenAI(
    api_key=API_KEY,
    base_url=url
)

print(f"\n✅ Подключение к GigaChat настроено")
print(f"   Base URL: {url}")
print()


✅ API ключ загружен успешно
   API_KEY: ZDk5MjY3MmMtM2Q4My00...

✅ Подключение к GigaChat настроено
   Base URL: https://foundation-models.api.cloud.ru/v1



## 🎓 Блок 2: Обучение модели на всех данных

**Первый запуск:** ~10-15 минут (подготовка датасета + индексация)
**Последующие:** ~2 секунды (загрузка готовой модели)

In [2]:
# ============================================================================
# ОБУЧЕНИЕ МОДЕЛИ (ИНДЕКСАЦИЯ)
# ============================================================================

print("\n" + "="*80)
print("🎓 ОБУЧЕНИЕ МОДЕЛИ (ИНДЕКСАЦИЯ ДАННЫХ)")
print("="*80)

import pickle
from datetime import datetime

class GigaChatFineTuningAnalyzer:
    """Анализатор с обучением на ВСЕХ данных"""
    
    def __init__(self, client):
        self.client = client
        self.training_examples = []
        self.regulations = []
        self.laws_corpus = []
        self.gigachat_prompts = []
        self.is_trained = False
        self.model_weights = None
        
    def load_training_data(self):
        """Загружает ВСЕ 4 источника данных"""
        print("\n🔄 Загрузка ВСЕХ обучающих данных...")
        
        # 1. Excel
        print("\n📊 [1/4] Загрузка dataset/train_5000.xlsx...")
        self.training_examples = self._load_excel()
        print(f"   ✅ Загружено примеров: {len(self.training_examples)}")
        
        # 2. Законы
        print("\n📄 [2/4] Загрузка data/regulations.json...")
        self.regulations = self._load_regulations()
        print(f"   ✅ Загружено статей: {len(self.regulations)}")
        
        # 3. Корпус
        print("\n📚 [3/4] Загрузка data/laws_corpus.json...")
        self.laws_corpus = self._load_laws_corpus()
        print(f"   ✅ Загружено документов: {len(self.laws_corpus)}")
        
        # 4. GigaChat промпты
        print("\n💬 [4/4] Загрузка data/gigachat_training.jsonl...")
        self.gigachat_prompts = self._load_gigachat_prompts()
        print(f"   ✅ Загружено промптов: {len(self.gigachat_prompts)}")
        
        total = len(self.training_examples) + len(self.regulations) + len(self.laws_corpus) + len(self.gigachat_prompts)
        print(f"\n" + "="*80)
        print(f"✅ ВСЕГО ЗАГРУЖЕНО: {total} документов")
        print(f"   • Примеры из Excel: {len(self.training_examples)}")
        print(f"   • Статьи законов: {len(self.regulations)}")
        print(f"   • Корпус документов: {len(self.laws_corpus)}")
        print(f"   • GigaChat промпты: {len(self.gigachat_prompts)}")
        print("="*80 + "\n")
        
        return total
    
    def _load_excel(self):
        excel_path = Path("dataset/train_5000.xlsx")
        if not excel_path.exists():
            return []
        
        df = pd.read_excel(excel_path)
        examples = []
        
        for _, row in df.iterrows():
            text = str(row.get('Текст', '')).strip()
            violation_type = str(row.get('Есть нарушение (Да/Нет/Потенциальное нарушение)', '')).strip()
            law = str(row.get('Закон', '')).strip()
            article = str(row.get('Статья', '')).strip()
            article_name = str(row.get('Название статьи', '')).strip()
            description = str(row.get('Выдержка/описание', '')).strip()
            risk = str(row.get('Риск', '')).strip()
            recommendation = str(row.get('Рекомендации (план по устранению)', '')).strip()
            
            if text and len(text) > 10:
                examples.append({
                    'text': text,
                    'violation_type': violation_type,
                    'law': law if law != 'nan' else '',
                    'article': article if article != 'nan' else '',
                    'article_name': article_name if article_name != 'nan' else '',
                    'description': description if description != 'nan' else '',
                    'risk': risk if risk != 'nan' else '',
                    'recommendation': recommendation if recommendation != 'nan' else ''
                })
        
        return examples
    
    def _load_regulations(self):
        json_path = Path("data/regulations.json")
        if not json_path.exists():
            return []
        
        regulations = []
        with open(json_path, 'r', encoding='utf-8') as f:
            for line in f:
                doc = json.loads(line)
                text = doc.get('text', '').strip()
                if text and len(text) > 30:
                    regulations.append({
                        'text': text,
                        'law_name': doc.get('law_name', ''),
                        'article_number': doc.get('article_number', '')
                    })
        
        return regulations
    
    def _load_laws_corpus(self):
        json_path = Path("data/laws_corpus.json")
        if not json_path.exists():
            return []
        
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                corpus = json.load(f)
            
            if isinstance(corpus, list):
                return [{'text': doc.get('text', ''), 'source': doc.get('source', '')} 
                        for doc in corpus if doc.get('text')]
            return []
        except:
            return []
    
    def _load_gigachat_prompts(self):
        jsonl_path = Path("data/gigachat_training.jsonl")
        if not jsonl_path.exists():
            return []
        
        prompts = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    line = line.strip()
                    if not line:
                        continue
                    
                    data = json.loads(line)
                    if 'messages' not in data:
                        continue
                    
                    text = ''
                    violation_data = {}
                    
                    for msg in data['messages']:
                        role = msg.get('role', '')
                        content = msg.get('content', '')
                        
                        if role == 'user' and 'Проанализируй рекламный текст:' in content:
                            start_idx = content.find('"')
                            end_idx = content.find('"', start_idx + 1)
                            if start_idx != -1 and end_idx != -1:
                                text = content[start_idx+1:end_idx]
                        
                        elif role == 'assistant':
                            try:
                                violation_data = json.loads(content)
                            except:
                                pass
                    
                    if text and len(text) > 5:
                        prompts.append({
                            'text': text,
                            'violation_data': violation_data
                        })
                
                except Exception:
                    continue
        
        return prompts
    
    def fine_tune(self):
        """Обучение (индексация) на ВСЕХ 4 источниках"""
        if not self.training_examples:
            print("❌ Нет данных для обучения!")
            return False
        
        print("\n" + "="*80)
        print("🎓 ЗАПУСК ИНДЕКСАЦИИ НА ВСЕХ 4 ИСТОЧНИКАХ!")
        print("="*80)
        
        total_docs = len(self.training_examples) + len(self.regulations) + len(self.laws_corpus) + len(self.gigachat_prompts)
        print(f"\n⏳ Будет обработано {total_docs} документов")
        print(f"⏳ Примерное время: ~{total_docs // 1000 + 1} минут\n")
        
        self.model_weights = {
            'examples_index': {},
            'patterns': {},
            'regulations_index': {},
            'laws_corpus_index': {},
            'gigachat_prompts_index': {},
            'trained_at': datetime.now().isoformat()
        }
        
        import time
        start_time = time.time()
        
        # ШАГ 1: Excel
        print("="*80)
        print("📊 ШАГ 1/4: Индексация примеров из Excel")
        print("="*80 + "\n")
        
        total = len(self.training_examples)
        for i, example in enumerate(self.training_examples, 1):
            if i % 500 == 0 or i == 1:
                progress = (i / total) * 100
                elapsed = time.time() - start_time
                eta = (elapsed / i) * (total - i) if i > 0 else 0
                print(f"   [{i}/{total}] {progress:.1f}% | Прошло: {elapsed:.0f}с | Осталось: ~{eta:.0f}с")
            
            text_lower = example['text'].lower()
            keywords = set(text_lower.split())
            
            for keyword in keywords:
                if keyword not in self.model_weights['examples_index']:
                    self.model_weights['examples_index'][keyword] = []
                self.model_weights['examples_index'][keyword].append(example)
            
            if example['violation_type'] in ['Да', 'Потенциальное нарушение']:
                pattern_key = f"{example['law']}_{example['article']}"
                if pattern_key not in self.model_weights['patterns']:
                    self.model_weights['patterns'][pattern_key] = []
                self.model_weights['patterns'][pattern_key].append(example)
        
        print(f"\n   ✅ Обработано {total} примеров")
        print(f"   ✅ Создано {len(self.model_weights['examples_index'])} ключевых слов")
        
        # ШАГ 2: Законы
        print("\n" + "="*80)
        print("📄 ШАГ 2/4: Индексация статей законов")
        print("="*80 + "\n")
        
        for i, reg in enumerate(self.regulations, 1):
            if i % 200 == 0 or i == 1:
                progress = (i / len(self.regulations)) * 100
                print(f"   [{i}/{len(self.regulations)}] {progress:.1f}%")
            
            text_lower = reg['text'].lower()
            keywords = set(text_lower.split())
            
            for keyword in keywords:
                if keyword not in self.model_weights['regulations_index']:
                    self.model_weights['regulations_index'][keyword] = []
                self.model_weights['regulations_index'][keyword].append(reg)
        
        print(f"\n   ✅ Обработано {len(self.regulations)} статей")
        print(f"   ✅ Создано {len(self.model_weights['regulations_index'])} ключевых слов")
        
        # ШАГ 3: Корпус
        print("\n" + "="*80)
        print("📚 ШАГ 3/4: Индексация корпуса документов")
        print("="*80 + "\n")
        
        for i, doc in enumerate(self.laws_corpus, 1):
            text_lower = doc['text'].lower()
            keywords = set(text_lower.split())
            
            for keyword in keywords:
                if keyword not in self.model_weights['laws_corpus_index']:
                    self.model_weights['laws_corpus_index'][keyword] = []
                self.model_weights['laws_corpus_index'][keyword].append(doc)
        
        print(f"   ✅ Обработано {len(self.laws_corpus)} документов")
        print(f"   ✅ Создано {len(self.model_weights['laws_corpus_index'])} ключевых слов")
        
        # ШАГ 4: GigaChat промпты
        print("\n" + "="*80)
        print("💬 ШАГ 4/4: Индексация GigaChat промптов")
        print("="*80 + "\n")
        
        for i, prompt in enumerate(self.gigachat_prompts, 1):
            if i % 500 == 0 or i == 1:
                progress = (i / len(self.gigachat_prompts)) * 100
                print(f"   [{i}/{len(self.gigachat_prompts)}] {progress:.1f}%")
            
            if prompt.get('text'):
                text_lower = prompt['text'].lower()
                keywords = set(text_lower.split())
                
                for keyword in keywords:
                    if keyword not in self.model_weights['gigachat_prompts_index']:
                        self.model_weights['gigachat_prompts_index'][keyword] = []
                    self.model_weights['gigachat_prompts_index'][keyword].append(prompt)
        
        print(f"\n   ✅ Обработано {len(self.gigachat_prompts)} промптов")
        print(f"   ✅ Создано {len(self.model_weights['gigachat_prompts_index'])} ключевых слов")
        
        # Сохраняем
        elapsed_total = time.time() - start_time
        print("\n" + "="*80)
        print("📊 ИТОГОВАЯ СТАТИСТИКА:")
        print("="*80)
        print(f"   • ВСЕГО обработано: {total_docs} документов")
        print(f"   • Время обучения: {elapsed_total:.1f} сек ({elapsed_total/60:.1f} мин)")
        print(f"   • Ключевых слов (примеры): {len(self.model_weights['examples_index'])}")
        print(f"   • Ключевых слов (законы): {len(self.model_weights['regulations_index'])}")
        print(f"   • Ключевых слов (корпус): {len(self.model_weights['laws_corpus_index'])}")
        print(f"   • Ключевых слов (промпты): {len(self.model_weights['gigachat_prompts_index'])}")
        
        print("\n💾 Сохранение модели...")
        weights_path = Path("data/model_weights.pkl")
        with open(weights_path, 'wb') as f:
            pickle.dump(self.model_weights, f)
        print(f"   ✅ Модель сохранена в {weights_path}")
        
        self.is_trained = True
        
        print("\n" + "="*80)
        print("✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!")
        print("="*80 + "\n")
        
        return True
    
    def load_trained_model(self):
        weights_path = Path("data/model_weights.pkl")
        
        if not weights_path.exists():
            return False
        
        print("📥 Загрузка обученной модели...")
        with open(weights_path, 'rb') as f:
            self.model_weights = pickle.load(f)
        
        self.is_trained = True
        
        print(f"✅ Модель загружена!")
        print(f"   • Обучена: {self.model_weights['trained_at']}")
        print(f"   • Ключевых слов: ~{len(self.model_weights.get('examples_index', {})) + len(self.model_weights.get('regulations_index', {})) + len(self.model_weights.get('laws_corpus_index', {})) + len(self.model_weights.get('gigachat_prompts_index', {}))}")
        print()
        
        return True
    
    # Метод analyze будет добавлен в следующей ячейке

# Создаём анализатор
analyzer = GigaChatFineTuningAnalyzer(client)

# Проверяем, есть ли уже обученная модель
if analyzer.load_trained_model():
    print("🚀 Модель уже обучена! Переходим к тестам.")
else:
    print("🔄 Модели нет. Начинаем полное обучение...\n")
    analyzer.load_training_data()
    analyzer.fine_tune()

elapsed_total = time.time() - start_total
print(f"\n⏱️  ОБЩЕЕ ВРЕМЯ: {elapsed_total:.1f} секунд ({elapsed_total/60:.1f} минут)")



🎓 ОБУЧЕНИЕ МОДЕЛИ (ИНДЕКСАЦИЯ ДАННЫХ)
📥 Загрузка обученной модели...
✅ Модель загружена!
   • Обучена: 2025-10-28T04:04:51.369417
   • Ключевых слов: ~35471

🚀 Модель уже обучена! Переходим к тестам.


NameError: name 'time' is not defined

# Добавляем метод analyze к классу
# ПОЛНЫЙ метод analyze с проверкой ВСЕХ нарушений (СТРОГИЕ ПРОВЕРКИ)
def analyze_method(self, text: str, resource_type: str = "Свой", resource: str = "Telegram") -> dict:
    """Анализ текста с ПОЛНОЙ проверкой ВСЕХ нарушений (ТОЛЬКО РЕАЛЬНЫЕ НАРУШЕНИЯ)"""
    if not self.is_trained or not self.model_weights:
        return {"error": "Модель не обучена!"}
    
    import re
    
    # ========================================================================
    # СОБИРАЕМ ВСЕ НАРУШЕНИЯ (НЕ ОСТАНАВЛИВАЕМСЯ НА ПЕРВОМ!)
    # ========================================================================
    all_violations = []  # Список всех найденных нарушений
    
    text_lower = text.lower()
    
    # ========================================================================
    # БЛОК 1: ОБЯЗАТЕЛЬНЫЕ ЭЛЕМЕНТЫ МАРКИРОВКИ
    # ========================================================================
    
    # 🔴 ПРОВЕРКА 1: Маркировка "Реклама"
    has_reklama = 'реклама' in text_lower and text_lower.count('"') % 2 == 0
    
    if not has_reklama:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 5, п. 7)",
            "article": "5.7",
            "article_name": "Маркировка рекламы",
            "description": "Отсутствует обязательная маркировка 'Реклама'",
            "risk": "Штраф для юр.лиц: 500 000 - 1 000 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Добавить слово 'Реклама:' в начале текста"
        })
    
    # 🔴 ПРОВЕРКА 2: ИНН (10 или 12 цифр)
    inn_found = []
    inn_explicit = re.findall(r'инн[\:\s\.]+(\d+)', text_lower)
    
    for inn_raw in inn_explicit:
        inn_digits = ''.join(c for c in inn_raw if c.isdigit())
        length = len(inn_digits)
        format_valid = length in [10, 12]
        inn_found.append({
            'raw': inn_raw,
            'digits': inn_digits,
            'length': length,
            'format_valid': format_valid
        })
    
    if not inn_found:
        ten_digits = re.findall(r'\b\d{10}\b', text)
        for num in ten_digits:
            inn_found.append({'raw': num, 'digits': num, 'length': 10, 'format_valid': True})
        
        if not inn_found:
            twelve_digits = re.findall(r'\b\d{12}\b', text)
            for num in twelve_digits:
                inn_found.append({'raw': num, 'digits': num, 'length': 12, 'format_valid': True})
    
    has_valid_inn = any(inn['format_valid'] for inn in inn_found)
    
    if not has_valid_inn:
        if inn_found:
            invalid_inn = inn_found[0]
            all_violations.append({
                "law": "ФЗ № 38 'О рекламе' (ст. 5, п. 7)",
                "article": "5.7",
                "article_name": "Идентификация рекламодателя",
                "description": f"ИНН имеет неверный формат: {invalid_inn['length']} цифр (требуется 10 или 12)",
                "risk": "Штраф для юр.лиц: 500 000 - 1 000 000 руб. (ст. 14.3 КоАП)",
                "recommendation": f"Заменить ИНН {invalid_inn['digits']} на корректный (10 или 12 цифр)"
            })
        else:
            all_violations.append({
                "law": "ФЗ № 38 'О рекламе' (ст. 5, п. 7)",
                "article": "5.7",
                "article_name": "Идентификация рекламодателя",
                "description": "Отсутствует ИНН рекламодателя",
                "risk": "Штраф для юр.лиц: 500 000 - 1 000 000 руб. (ст. 14.3 КоАП)",
                "recommendation": "Добавить ИНН рекламодателя (10 цифр для юрлица, 12 для ИП)"
            })
    
    # 🔴 ПРОВЕРКА 3: erid токен
    erid_pattern = r'erid[\:\s]+([a-zA-Z0-9_\-]+)'
    erid_match = re.search(erid_pattern, text_lower)
    
    has_valid_erid = False
    erid_token = None
    
    if erid_match:
        erid_token = erid_match.group(1)
        if len(erid_token) >= 5:
            has_valid_erid = True
    
    if not has_valid_erid:
        if 'erid' in text_lower and not erid_token:
            desc = "Присутствует слово 'erid', но отсутствует сам токен"
        elif 'erid' in text_lower and erid_token and len(erid_token) < 5:
            desc = f"Токен erid слишком короткий: '{erid_token}' (минимум 5 символов)"
        else:
            desc = "Отсутствует токен erid"
        
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 18.1)",
            "article": "18.1",
            "article_name": "Маркировка интернет-рекламы",
            "description": desc,
            "risk": "Штраф для юр.лиц: 500 000 - 1 000 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Добавить уникальный токен erid (получите в рекламном кабинете)"
        })
    
    # ========================================================================
    # БЛОК 2: СОДЕРЖАНИЕ РЕКЛАМЫ (СТРОГИЕ ПРОВЕРКИ С ГРАНИЦАМИ СЛОВ)
    # ========================================================================
    
    # 🔴 ПРОВЕРКА 4: Непристойные слова (СТРОГАЯ - ТОЛЬКО ЦЕЛЫЕ СЛОВА!)
    obscene_patterns = [
        r'\bчлен\b',  # только целое слово "член"
        r'\bхуй\b',
        r'\bпизд[а-яёА-ЯЁ]*\b',  # пизда, пиздец и т.д.
        r'\bебать\b',
        r'\bебл[а-яёА-ЯЁ]*\b',
        r'\bбля[тдь]\b',
        r'\bсука\b',
        r'\bпидор\b',
        r'\bхер\b',
        r'\bжопа\b',
        r'\bгандон\b',
        r'\bдерьм[а-яёА-ЯЁ]*\b',
        r'\bговн[а-яёА-ЯЁ]*\b'
    ]
    
    found_obscene = []
    for pattern in obscene_patterns:
        matches = re.findall(pattern, text_lower)
        found_obscene.extend(matches)
    
    if found_obscene:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 5)",
            "article": "5",
            "article_name": "Общие требования к рекламе",
            "description": f"Содержит непристойные слова: {', '.join(set(found_obscene))}",
            "risk": "Штраф для юр.лиц: 100 000 - 500 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Удалить все непристойные слова"
        })
    
    # 🔴 ПРОВЕРКА 5: Недостоверные заявления (СТРОГАЯ)
    false_claims_patterns = [
        r'\bлучший\b',
        r'\bсамый лучший\b',
        r'\bномер 1\b',
        r'\b№\s*1\b',
        r'\bпервый в мире\b',
        r'\bединственный\b',
        r'\bгарантированный результат\b',
        r'\bгарантируем\b',
        r'\b100%\s*результат\b',
        r'\b100%\s*гарантия\b',
        r'\bбез рисков\b'
    ]
    
    found_false = []
    for pattern in false_claims_patterns:
        if re.search(pattern, text_lower):
            match = re.search(pattern, text_lower)
            if match:
                found_false.append(match.group(0))
    
    if found_false:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 5)",
            "article": "5",
            "article_name": "Общие требования к рекламе (достоверность)",
            "description": f"Недостоверные заявления без доказательств: {', '.join(set(found_false))}",
            "risk": "Штраф для юр.лиц: 100 000 - 500 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Удалить недостоверные заявления или предоставить доказательства"
        })
    
    # 🔴 ПРОВЕРКА 6: Розыгрыши без правил (СТРОГАЯ)
    lottery_patterns = [
        r'\bрозыгрыш\b',
        r'\bлотере[а-яёА-ЯЁ]*\b',
        r'\bвыиграть\b',
        r'\bприз\b',
        r'\bконкурс\b'
    ]
    
    has_lottery = any(re.search(pattern, text_lower) for pattern in lottery_patterns)
    has_rules = re.search(r'\b(правила|условия|положение)\b', text_lower)
    
    if has_lottery and not has_rules:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 9)",
            "article": "9",
            "article_name": "Особенности рекламы при проведении стимулирующих мероприятий",
            "description": "Стимулирующая акция без полных правил проведения",
            "risk": "Штраф для юр.лиц: 200 000 - 500 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Добавить ссылку на полные правила акции (сроки, порядок, условия)"
        })
    
    # 🔴 ПРОВЕРКА 7: Алкоголь (СТРОГАЯ - ТОЛЬКО ЯВНЫЕ УПОМИНАНИЯ!)
    alcohol_patterns = [
        r'\bпиво\b',
        r'\bвино\b',
        r'\bводка\b',
        r'\bвиски\b',
        r'\bконьяк\b',
        r'\bшампанское\b',
        r'\bалкоголь\b',
        r'\bспиртн[а-яёА-ЯЁ]*\b',
        r'\bликёр\b',
        r'\bром\b',
        r'\bтекила\b',
        r'\bджин\b'
    ]
    
    has_alcohol = any(re.search(pattern, text_lower) for pattern in alcohol_patterns)
    
    if has_alcohol:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 21)",
            "article": "21",
            "article_name": "Особенности рекламы алкогольной продукции",
            "description": "Реклама алкогольной продукции (требуется проверка возрастных ограничений 18+)",
            "risk": "Штраф для юр.лиц: 500 000 - 1 000 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Убедиться в наличии маркировки 18+ и соблюдении ограничений размещения"
        })
    
    # 🔴 ПРОВЕРКА 8: Табак (СТРОГАЯ)
    tobacco_patterns = [
        r'\bсигарет[а-яёА-ЯЁ]*\b',
        r'\bтабак\b',
        r'\bкальян\b',
        r'\bвейп\b',
        r'\bэлектронн[а-яёА-ЯЁ]*\s+сигарет[а-яёА-ЯЁ]*\b',
        r'\bснюс\b',
        r'\bнасвай\b'
    ]
    
    has_tobacco = any(re.search(pattern, text_lower) for pattern in tobacco_patterns)
    
    if has_tobacco:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 20)",
            "article": "20",
            "article_name": "Реклама табака и табачных изделий",
            "description": "Реклама табака и табачных изделий ЗАПРЕЩЕНА",
            "risk": "Штраф для юр.лиц: 1 000 000 - 1 500 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Удалить рекламу табачной продукции ПОЛНОСТЬЮ"
        })
    
    # 🔴 ПРОВЕРКА 9: Наркотики (СТРОГАЯ - ТОЛЬКО ЯВНЫЕ УПОМИНАНИЯ!)
    drugs_patterns = [
        r'\bмарихуан[а-яёА-ЯЁ]*\b',
        r'\bконопл[а-яёА-ЯЁ]*\b',
        r'\bгашиш\b',
        r'\bгероин\b',
        r'\bкокаин\b',
        r'\bспайс\b',
        r'\bамфетамин\b',
        r'\bметамфетамин\b',
        r'\bэкстази\b',
        r'\bлсд\b',
        r'\bпсихотроп[а-яёА-ЯЁ]*\b',
        r'\bнаркотик[а-яёА-ЯЁ]*\b'
    ]
    
    has_drugs = any(re.search(pattern, text_lower) for pattern in drugs_patterns)
    
    if has_drugs:
        all_violations.append({
            "law": "УК РФ (ст. 228)",
            "article": "228",
            "article_name": "Незаконный оборот наркотических средств",
            "description": "ПРОПАГАНДА НАРКОТИКОВ - УГОЛОВНОЕ ПРЕСТУПЛЕНИЕ",
            "risk": "УГОЛОВНАЯ ОТВЕТСТВЕННОСТЬ + штраф до 5 000 000 руб.",
            "recommendation": "НЕМЕДЛЕННО удалить весь контент, связанный с наркотиками"
        })
    
    # 🔴 ПРОВЕРКА 10: Лекарства и БАДы (СТРОГАЯ)
    medicine_patterns = [
        r'\bлечит\b',
        r'\bизлечивает\b',
        r'\bвылечит\b',
        r'\bисцеляет\b'
    ]
    
    has_medicine_claims = any(re.search(pattern, text_lower) for pattern in medicine_patterns)
    has_bad = re.search(r'\bбад\b', text_lower)
    has_medicine = re.search(r'\b(лекарство|препарат|таблетк[а-яёА-ЯЁ]*|капсул[а-яёА-ЯЁ]*)\b', text_lower)
    has_disclaimer = re.search(r'\b(противопоказан|консультация специалиста|не является лекарственным средством)\b', text_lower)
    
    if has_medicine_claims and not has_disclaimer:
        if has_bad:
            all_violations.append({
                "law": "ФЗ № 38 'О рекламе' (ст. 25)",
                "article": "25",
                "article_name": "Реклама биологически активных добавок",
                "description": "Реклама БАД с лечебными обещаниями без предупреждения",
                "risk": "Штраф для юр.лиц: 200 000 - 500 000 руб. (ст. 14.3 КоАП)",
                "recommendation": "Добавить текст: 'Не является лекарственным средством. Перед применением проконсультируйтесь со специалистом.'"
            })
        elif has_medicine:
            all_violations.append({
                "law": "ФЗ № 38 'О рекламе' (ст. 24)",
                "article": "24",
                "article_name": "Реклама лекарственных средств",
                "description": "Реклама лекарственного средства без обязательного предупреждения о противопоказаниях",
                "risk": "Штраф для юр.лиц: 300 000 - 700 000 руб. (ст. 14.3 КоАП)",
                "recommendation": "Добавить текст: 'Есть противопоказания. Необходима консультация специалиста.'"
            })
    
    # 🔴 ПРОВЕРКА 11: Персональные данные (СТРОГАЯ)
    personal_data_patterns = [
        r'\bвведите\s+(телефон|номер|email|почту|данные)\b',
        r'\bукажите\s+(телефон|номер|email|почту|данные)\b',
        r'\bоставьте\s+контакты\b',
        r'\bзаполните\s+форму\b',
        r'\bрегистрация\b'
    ]
    
    has_personal_data = any(re.search(pattern, text_lower) for pattern in personal_data_patterns)
    has_policy = re.search(r'\b(политика конфиденциальности|согласие на обработку|обработка персональных данных)\b', text_lower)
    
    if has_personal_data and not has_policy:
        all_violations.append({
            "law": "ФЗ № 152 'О персональных данных' (ст. 6)",
            "article": "6",
            "article_name": "Условия обработки персональных данных",
            "description": "Сбор персональных данных без ссылки на политику конфиденциальности и согласия",
            "risk": "Штраф для юр.лиц: 60 000 - 150 000 руб. (ст. 13.11 КоАП)",
            "recommendation": "Добавить ссылку на политику конфиденциальности и форму согласия на обработку ПД"
        })
    
    # ========================================================================
    # БЛОК 3: ПОИСК ДОПОЛНИТЕЛЬНЫХ НАРУШЕНИЙ ИЗ ОБУЧАЮЩИХ ДАННЫХ
    # ========================================================================
    
    keywords = set(text_lower.split())
    similar_examples = []
    
    for keyword in keywords:
        if keyword in self.model_weights.get('examples_index', {}):
            similar_examples.extend(self.model_weights['examples_index'][keyword][:10])
    
    # Проверяем похожие примеры
    for ex in similar_examples:
        if ex.get('violation_type') in ['Да', 'Потенциальное нарушение']:
            # Проверяем, не добавили ли мы уже это нарушение
            law = ex.get('law', '').strip()
            article = ex.get('article', '').strip()
            description = ex.get('description', '').strip()
            
            # Если это уникальное нарушение - добавляем
            is_duplicate = False
            for v in all_violations:
                if v['law'] == law and v['article'] == article:
                    is_duplicate = True
                    break
            
            if not is_duplicate and description:
                all_violations.append({
                    "law": law,
                    "article": article,
                    "article_name": ex.get('article_name', ''),
                    "description": description,
                    "risk": ex.get('risk', ''),
                    "recommendation": ex.get('recommendation', '')
                })
    
    # ========================================================================
    # ФОРМИРОВАНИЕ ИТОГОВОГО РЕЗУЛЬТАТА
    # ========================================================================
    
    if all_violations:
        # Объединяем все нарушения
        numbered_violations = [f"{i+1}) {v['description']}" for i, v in enumerate(all_violations)]
        
        return {
            "violation_status": "Да",
            "law": "; ".join([v['law'] for v in all_violations]),
            "article": "; ".join([v['article'] for v in all_violations]),
            "article_name": "; ".join([v['article_name'] for v in all_violations]),
            "description": "; ".join(numbered_violations),
            "risk": "; ".join([v['risk'] for v in all_violations]),
            "recommendation": "; ".join([f"{i+1}) {v['recommendation']}" for i, v in enumerate(all_violations)]),
            "inn_found": inn_found,
            "original_text": text,
            "total_violations": len(all_violations),
            "auto_detected": True
        }
    
    # Если не нашли нарушений
    return {
        "violation_status": "Нет",
        "law": "",
        "article": "",
        "article_name": "",
        "description": "✅ Нарушений не обнаружено. Проверены все блоки: маркировка, содержание, достоверность, стимулирующие мероприятия, персональные данные, алкоголь, табак, наркотики, лекарства.",
        "risk": "",
        "recommendation": "",
        "inn_found": inn_found,
        "original_text": text,
        "total_violations": 0,
        "matched_from_training": False
    }



GigaChatFineTuningAnalyzer.analyze = analyze_method


In [ ]:
# Добавляем метод analyze к классу
# ПОЛНЫЙ метод analyze с проверкой ВСЕХ нарушений
def analyze_method(self, text: str, resource_type: str = "Свой", resource: str = "Telegram") -> dict:
    """Анализ текста с ПОЛНОЙ проверкой ВСЕХ нарушений"""
    if not self.is_trained or not self.model_weights:
        return {"error": "Модель не обучена!"}
    
    import re
    
    # ========================================================================
    # СОБИРАЕМ ВСЕ НАРУШЕНИЯ (НЕ ОСТАНАВЛИВАЕМСЯ НА ПЕРВОМ!)
    # ========================================================================
    all_violations = []  # Список всех найденных нарушений
    
    text_lower = text.lower()
    inn_found = []  # Инициализация для совместимости с return
    
    
    # ========================================================================
    # БЛОК 1: ОБЯЗАТЕЛЬНЫЕ ЭЛЕМЕНТЫ МАРКИРОВКИ
    # ========================================================================
    
    # 🔴 ПРОВЕРКА 1: Маркировка "Реклама"
    has_reklama = 'реклама' in text_lower and text_lower.count('"') % 2 == 0
    
    if not has_reklama:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 5, п. 7)",
            "article": "5.7",
            "article_name": "Маркировка рекламы",
            "description": "Отсутствует обязательная маркировка 'Реклама'",
            "risk": "Штраф для юр.лиц: 500 000 - 1 000 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Добавить слово 'Реклама:' в начале текста"
        })
    
    # 🔴 ПРОВЕРКА 2: ИНН (УПРОЩЁННАЯ: если есть слово "ИНН" - OK)
    inn_found = []  # Для совместимости с return
    has_inn_word = re.search(r'\bинн\b', text_lower)
    
    if not has_inn_word:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 5, п. 7)",
            "article": "5.7",
            "article_name": "Идентификация рекламодателя",
            "description": "Отсутствует ИНН рекламодателя",
            "risk": "Штраф для юр.лиц: 500 000 - 1 000 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Добавить ИНН рекламодателя"
        })

    # 🔴 ПРОВЕРКА 3: erid (УПРОЩЁННАЯ: если есть слово "erid" - OK)
    has_erid_word = re.search(r'\berid\b', text_lower)
    
    if not has_erid_word:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 18.1)",
            "article": "18.1",
            "article_name": "Маркировка интернет-рекламы",
            "description": "Отсутствует токен erid",
            "risk": "Штраф для юр.лиц: 500 000 - 1 000 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Добавить уникальный токен erid (получите в рекламном кабинете)"
        })

    erid_pattern = r'erid[:\s]+([a-zA-Z0-9_\-]+)'
    erid_match = re.search(erid_pattern, text_lower)
    
    has_valid_erid = False
    erid_token = None
    
    if erid_match:
        erid_token = erid_match.group(1)
        if len(erid_token) >= 5:
            has_valid_erid = True
    
    if not has_valid_erid:
        if 'erid' in text_lower and not erid_token:
            desc = "Присутствует слово 'erid', но отсутствует сам токен"
        elif 'erid' in text_lower and erid_token and len(erid_token) < 5:
            desc = f"Токен erid слишком короткий: '{erid_token}' (минимум 5 символов)"
        else:
            desc = "Отсутствует токен erid"
        
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 18.1)",
            "article": "18.1",
            "article_name": "Маркировка интернет-рекламы",
            "description": desc,
            "risk": "Штраф для юр.лиц: 500 000 - 1 000 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Добавить уникальный токен erid (получите в рекламном кабинете)"
        })
    
    # ========================================================================
    # БЛОК 2: СОДЕРЖАНИЕ РЕКЛАМЫ
    # ========================================================================
    
    # 🔴 ПРОВЕРКА 4: Непристойные слова
    obscene_words = ['член', 'хуй', 'пизд', 'ебать', 'ебл', 'бля', 'сука', 'пидор', 'хер', 'жопа', 'гандон', 'дерьм', 'говн', 'уеб', 'ебан', 'блять']
    found_obscene = [word for word in obscene_words if word in text_lower]
    
    if found_obscene:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 5)",
            "article": "5",
            "article_name": "Общие требования к рекламе",
            "description": f"Содержит непристойные слова: {', '.join(set(found_obscene))}",
            "risk": "Штраф для юр.лиц: 100 000 - 500 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Удалить все непристойные слова"
        })
    
    # 🔴 ПРОВЕРКА 5: Недостоверные заявления
    false_claims = ['лучший', 'самый лучший', 'номер 1', 'первый в мире', 'единственный', 'гарантированный результат', 'гарантируем', '100% результат', '100% гарантия', 'без рисков']
    found_false = [claim for claim in false_claims if claim in text_lower]
    
    if found_false:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 5)",
            "article": "5",
            "article_name": "Общие требования к рекламе (достоверность)",
            "description": f"Недостоверные заявления без доказательств: {', '.join(set(found_false))}",
            "risk": "Штраф для юр.лиц: 100 000 - 500 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Удалить недостоверные заявления или предоставить доказательства"
        })
    
    # 🔴 ПРОВЕРКА 6: Розыгрыши без правил
    lottery_keywords = ['розыгрыш', 'лотере', 'выиграть', 'приз', 'подарок', 'конкурс', 'получи шанс']
    has_lottery = any(word in text_lower for word in lottery_keywords)
    has_rules = any(word in text_lower for word in ['правила', 'условия', 'положение'])
    
    if has_lottery and not has_rules:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 9)",
            "article": "9",
            "article_name": "Особенности рекламы при проведении стимулирующих мероприятий",
            "description": "Стимулирующая акция без полных правил проведения",
            "risk": "Штраф для юр.лиц: 200 000 - 500 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Добавить ссылку на полные правила акции (сроки, порядок, условия)"
        })
    
    # 🔴 ПРОВЕРКА 7: Алкоголь
    alcohol_keywords = ['пиво', 'вино', 'водка', 'виски', 'коньяк', 'шампанское', 'алкогол', 'спирт', 'ликёр', 'ром', 'текила', 'джин']
    has_alcohol = any(word in text_lower for word in alcohol_keywords)
    
    if has_alcohol:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 21)",
            "article": "21",
            "article_name": "Особенности рекламы алкогольной продукции",
            "description": "Реклама алкогольной продукции (требуется проверка возрастных ограничений 18+)",
            "risk": "Штраф для юр.лиц: 500 000 - 1 000 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Убедиться в наличии маркировки 18+ и соблюдении ограничений размещения"
        })
    
    # 🔴 ПРОВЕРКА 8: Табак
    tobacco_keywords = ['сигарет', 'табак', 'кальян', 'вейп', 'электронные сигареты', 'снюс', 'насвай']
    has_tobacco = any(word in text_lower for word in tobacco_keywords)
    
    if has_tobacco:
        all_violations.append({
            "law": "ФЗ № 38 'О рекламе' (ст. 20)",
            "article": "20",
            "article_name": "Реклама табака и табачных изделий",
            "description": "Реклама табака и табачных изделий ЗАПРЕЩЕНА",
            "risk": "Штраф для юр.лиц: 1 000 000 - 1 500 000 руб. (ст. 14.3 КоАП)",
            "recommendation": "Удалить рекламу табачной продукции ПОЛНОСТЬЮ"
        })
    
    # 🔴 ПРОВЕРКА 9: Наркотики
    drugs_keywords = ['марихуан', 'конопл', 'гашиш', 'героин', 'кокаин', 'спайс', 'амфетамин', 'метамфетамин', 'экстази', 'лсд', 'психотроп']
    has_drugs = any(word in text_lower for word in drugs_keywords)
    
    if has_drugs:
        all_violations.append({
            "law": "УК РФ (ст. 228)",
            "article": "228",
            "article_name": "Незаконный оборот наркотических средств",
            "description": "ПРОПАГАНДА НАРКОТИКОВ - УГОЛОВНОЕ ПРЕСТУПЛЕНИЕ",
            "risk": "УГОЛОВНАЯ ОТВЕТСТВЕННОСТЬ + штраф до 5 000 000 руб.",
            "recommendation": "НЕМЕДЛЕННО удалить весь контент, связанный с наркотиками"
        })
    
    # 🔴 ПРОВЕРКА 10: Лекарства и БАДы
    medicine_keywords = ['лечит', 'излечивает', 'вылечит', 'исцеляет', 'бад', 'лекарство', 'препарат', 'таблетк', 'капсул', 'лечение']
    has_medicine = any(word in text_lower for word in medicine_keywords)
    has_disclaimer = any(word in text_lower for word in ['противопоказан', 'консультация специалиста', 'не является лекарственным средством'])
    
    if has_medicine and not has_disclaimer:
        if 'бад' in text_lower:
            all_violations.append({
                "law": "ФЗ № 38 'О рекламе' (ст. 25)",
                "article": "25",
                "article_name": "Реклама биологически активных добавок",
                "description": "Реклама БАД без обязательного предупреждения 'Не является лекарственным средством'",
                "risk": "Штраф для юр.лиц: 200 000 - 500 000 руб. (ст. 14.3 КоАП)",
                "recommendation": "Добавить текст: 'Не является лекарственным средством. Перед применением проконсультируйтесь со специалистом.'"
            })
        else:
            all_violations.append({
                "law": "ФЗ № 38 'О рекламе' (ст. 24)",
                "article": "24",
                "article_name": "Реклама лекарственных средств",
                "description": "Реклама лекарственного средства без обязательного предупреждения о противопоказаниях",
                "risk": "Штраф для юр.лиц: 300 000 - 700 000 руб. (ст. 14.3 КоАП)",
                "recommendation": "Добавить текст: 'Есть противопоказания. Необходима консультация специалиста.'"
            })
    
    # 🔴 ПРОВЕРКА 11: Персональные данные
    personal_data_keywords = ['введите телефон', 'укажите email', 'ваш номер', 'ваша почта', 'оставьте контакты', 'заполните форму', 'регистрация', 'введите данные']
    has_personal_data = any(word in text_lower for word in personal_data_keywords)
    has_policy = any(word in text_lower for word in ['политика конфиденциальности', 'согласие на обработку', 'обработка персональных данных'])
    
    if has_personal_data and not has_policy:
        all_violations.append({
            "law": "ФЗ № 152 'О персональных данных' (ст. 6)",
            "article": "6",
            "article_name": "Условия обработки персональных данных",
            "description": "Сбор персональных данных без ссылки на политику конфиденциальности и согласия",
            "risk": "Штраф для юр.лиц: 60 000 - 150 000 руб. (ст. 13.11 КоАП)",
            "recommendation": "Добавить ссылку на политику конфиденциальности и форму согласия на обработку ПД"
        })
    
    # ========================================================================
    # БЛОК 3: ПОИСК ДОПОЛНИТЕЛЬНЫХ НАРУШЕНИЙ ИЗ ОБУЧАЮЩИХ ДАННЫХ
    # ========================================================================
    
    keywords = set(text_lower.split())
    similar_examples = []
    
    for keyword in keywords:
        if keyword in self.model_weights.get('examples_index', {}):
            similar_examples.extend(self.model_weights['examples_index'][keyword][:10])
    
    # Проверяем похожие примеры
    for ex in similar_examples:
        if ex.get('violation_type') in ['Да', 'Потенциальное нарушение']:
            # Проверяем, не добавили ли мы уже это нарушение
            law = ex.get('law', '').strip()
            article = ex.get('article', '').strip()
            description = ex.get('description', '').strip()
            
            # Если это уникальное нарушение - добавляем
            is_duplicate = False
            for v in all_violations:
                if v['law'] == law and v['article'] == article:
                    is_duplicate = True
                    break
            
            if not is_duplicate and description:
                all_violations.append({
                    "law": law,
                    "article": article,
                    "article_name": ex.get('article_name', ''),
                    "description": description,
                    "risk": ex.get('risk', ''),
                    "recommendation": ex.get('recommendation', '')
                })
    
    # ========================================================================
    # ФОРМИРОВАНИЕ ИТОГОВОГО РЕЗУЛЬТАТА
    # ========================================================================
    
    if all_violations:
        # Объединяем все нарушения
        numbered_violations = [f"{i+1}) {v['description']}" for i, v in enumerate(all_violations)]
        
        return {
            "violation_status": "Да",
            "law": "; ".join([v['law'] for v in all_violations]),
            "article": "; ".join([v['article'] for v in all_violations]),
            "article_name": "; ".join([v['article_name'] for v in all_violations]),
            "description": "; ".join(numbered_violations),
            "risk": "; ".join([v['risk'] for v in all_violations]),
            "recommendation": "; ".join([f"{i+1}) {v['recommendation']}" for i, v in enumerate(all_violations)]),
            "inn_found": inn_found,
            "original_text": text,
            "total_violations": len(all_violations),
            "auto_detected": True
        }
    
    # Если не нашли нарушений
    return {
        "violation_status": "Нет",
        "law": "",
        "article": "",
        "article_name": "",
        "description": "✅ Нарушений не обнаружено. Проверены все блоки: маркировка, содержание, достоверность, стимулирующие мероприятия, персональные данные, алкоголь, табак, наркотики, лекарства.",
        "risk": "",
        "recommendation": "",
        "inn_found": inn_found,
        "original_text": text,
        "total_violations": 0,
        "matched_from_training": False
    }



GigaChatFineTuningAnalyzer.analyze = analyze_method

# ============================================================================
# ТЕСТЫ
# ============================================================================

test_cases = [
    {
        "name": "🚨 ТЕСТ 1: 💼 Хочешь узнать, подходит ли тебе новая профессия Data Analyst? Пройди бесплатный тест и получи персональные рекомендации 👉 [пример_ссылки]",
        "text": "Все размеры в наличии!"
    },
    {
        "name": "🚨 ТЕСТ 2: Недостоверная реклама",
        "text": "Товар имеет знак РСТ"
    },
    {
        "name": "🚨 ТЕСТ 3: Непристойное слово",
        "text": "член"
    },
    {
        "name": "✅ ТЕСТ 4: Легальная реклама",
        "text": "Купите новый iPhone 15 Pro со скидкой 10%!"
    },
]

print("\n" + "="*80)
print("🧪 АВТОМАТИЧЕСКОЕ ТЕСТИРОВАНИЕ")
print("="*80 + "\n")

for test in test_cases:
    print(f"{'='*80}")
    print(f"{test['name']}")
    print(f"{'='*80}")
    print(f"\n📝 Текст: {test['text']}")
    print(f"\n🔍 Анализ...")
    
    result = analyzer.analyze(text=test['text'])
    
    print("\n📊 РЕЗУЛЬТАТ:")
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print()

print("\n" + "="*80)
print("✅ ВСЕ ТЕСТЫ ЗАВЕРШЕНЫ!")
print("="*80 + "\n")



🧪 АВТОМАТИЧЕСКОЕ ТЕСТИРОВАНИЕ

🚨 ТЕСТ 1: 💼 Хочешь узнать, подходит ли тебе новая профессия Data Analyst? Пройди бесплатный тест и получи персональные рекомендации 👉 [пример_ссылки]

📝 Текст: Все размеры в наличии!

🔍 Анализ...

📊 РЕЗУЛЬТАТ:
{
  "violation_status": "Да",
  "law": "ФЗ № 38 'О рекламе' (ст. 5, п. 7); ФЗ № 38 'О рекламе' (ст. 5, п. 7); ФЗ № 38 'О рекламе' (ст. 18.1); ФЗ № 38 'О рекламе' (ст. 18.1); ФЗ-38; ФЗ-3",
  "article": "5.7; 5.7; 18.1; 18.1; 5; 7",
  "article_name": "Маркировка рекламы; Идентификация рекламодателя; Маркировка интернет-рекламы; Маркировка интернет-рекламы; Общие требования к рекламе; Запрет пропаганды наркотических средств",
  "description": "1) Отсутствует обязательная маркировка 'Реклама'; 2) Отсутствует ИНН рекламодателя; 3) Отсутствует токен erid; 4) Отсутствует токен erid; 5) Реклама должна быть добросовестной и достоверной. Недопустима недостоверная реклама.; 6) Запрещена пропаганда/содействие обороту наркотических средств.",
  "risk": "Штраф д

In [ ]:
# ============================================================================
# 🎯 ТЕСТ - УМНАЯ КЛАССИФИКАЦИЯ VIOLATION + GIGA-CHECK (ДЕТАЛЬНЫЕ ПРАВИЛА)
# ============================================================================

my_text = """
🎭 **Театральные и культурные события**: Узнай о спектаклях, выставках, концертах и фестивалях. Мы делаем анонсы, подборки лучших событий и рекомендации, чтобы ты всегда был в центре культурной жизни. Читай интервью с артистами, обзоры постановок и получай вдохновение для посещения культурных мероприятий.  Реклама. ООО АААА.. ИНН 00000000. ERID 
"""

# Анализ
result = analyzer.analyze(text=my_text.strip())

# ============================================================================
# УМНАЯ ПОСТ-ОБРАБОТКА (YES / MAYBE / NO)
# ============================================================================

import re

def decide_violation(text: str, result: dict) -> str:
    """
    Умная классификация нарушений с ДЕТАЛЬНЫМИ ПРАВИЛАМИ:
    - YES: жёсткие запреты + недостоверность с доказательствами
    - MAYBE: формальные требования + неоднозначные случаи
    - NO: нет нарушений
    """
    
    if result.get("violation_status") == "Нет":
        return "no"
    
    text_lower = text.lower()
    description_lower = result.get('description', '').lower()
    
    # ========================================================================
    # 🔴 ЖЁСТКИЕ ЗАПРЕТЫ (YES) - РАСШИРЕННЫЙ СПИСОК
    # ========================================================================
    
    hard_triggers = [
        # 1) Наркотики, оружие, экстремизм
        (r'\b(наркотик|травк|кокаин|героин|амфетамин|марихуан|конопл|спайс|психотроп)\b', 
         'наркотики - жёсткий запрет'),
        (r'\b(оружие|пистолет|автомат|патрон)\b', 
         'оружие - жёсткий запрет'),
        (r'\b(экстремизм|ненависть|терроризм)\b', 
         'экстремизм - жёсткий запрет'),
        
        # 2) БАД/Лекарства с лечебными обещаниями
        (r'\bбад\b.*\b(лечит|вылечит|излечивает|исцеляет)\b', 
         'БАД с лечебными обещаниями - жёсткий запрет'),
        (r'\b(лечит|вылечит|излечивает|исцеляет)\b.*\bбад\b', 
         'БАД с лечебными обещаниями - жёсткий запрет'),
        
        # 3) Гарантированный доход/прибыль
        (r'\b(гарантир(уем|ованн)|100%)\b.*\b(доход|прибыл|результат)\b', 
         'гарантированный доход - жёсткий запрет'),
        (r'\bвложи.*получи.*гарантируем\b', 
         'финансовая пирамида - жёсткий запрет'),
        
        # 4) Табак (полный запрет)
        (r'\b(сигарет|табак|вейп|снюс|насвай)\b', 
         'табак - полный запрет'),
        
        # 5) Порнография
        (r'\b(порно|xxx|секс-услуг)\b', 
         'порнография - жёсткий запрет'),
        
        # 6) Непристойные слова
        (r'\b(член|хуй|пизд|ебать|ебл)\b', 
         'непристойные слова - жёсткий запрет'),
        
        # 7) Недостоверность с явными ложными заявлениями
        (r'\b(продано более|более миллиона)\b.*\b(упаковок|товаров)\b', 
         'недостоверное утверждение о продажах'),
        (r'\b(мгновенная доставка|доставка за час)\b', 
         'недостоверное утверждение о доставке'),
        (r'\bцена снижена на (80|90|95)%\b', 
         'манипуляция ценой - ложная скидка'),
        (r'\b(единственный|единственная)\b.*\b(школа|компания|магазин)\b', 
         'недостоверное утверждение "единственный"'),
        
        # 8) Порочащие сравнения с конкурентами
        (r'\b(их сервис|их товар)\b.*\b(издевательство|кошмар|ужас|подделк)\b', 
         'порочащие утверждения о конкурентах'),
        (r'\b(остальные|другие)\b.*\b(подделк|развалятся|дешёвые)\b', 
         'порочащие утверждения о конкурентах'),
        
        # 9) Призыв к незаконным действиям
        (r'\b(отомстить|найдём компромат|взлом|взломаем)\b', 
         'призыв к противоправным действиям'),
        
        # 10) Детская аудитория + опасный контент
        (r'\b(детский чай|детское питание)\b.*\b(лечит|вылечит)\b', 
         'детская аудитория + лечебные обещания'),
        
        # 11) Ложные официальные статусы
        (r'\b(одобрено воз|рекомендовано воз)\b', 
         'ложная ссылка на ВОЗ'),
        
        # 12) Медицинские функции у не-медизделий
        (r'\bмассажёр\b.*\b(мрт|выполняет функции)\b', 
         'приписывание медфункций не-медизделию'),
        (r'\bсвет.*\b(лечит|восстановление зрения)\b', 
         'лечебные обещания без статуса ЛС'),
        
        # 13) Обмен без вопросов (ввод в заблуждение)
        (r'\bобмен.*без вопросов\b', 
         'ввод в заблуждение об условиях обмена'),
        
        # 14) Акции без условий (явный ввод в заблуждение)
        (r'\b(выиграй|получи)\b.*\bбез условий\b', 
         'акция без существенных условий - ввод в заблуждение'),
    ]
    
    for pattern, reason in hard_triggers:
        if re.search(pattern, text_lower) or re.search(pattern, description_lower):
            return "yes"
    
    # ========================================================================
    # 🟡 ФОРМАЛЬНЫЕ ТРЕБОВАНИЯ И НЕОДНОЗНАЧНЫЕ СЛУЧАИ (MAYBE)
    # ========================================================================
    
    # Если все нарушения - только формальные (маркировка, ИНН, ERID, условия)
    formal_keywords = [
        'маркировка', 'реклама', 'инн', 'erid', 'токен',
        'существенные условия', 'условия акции', 'правила акции',
        'идентификация рекламодателя', 'маркировка интернет-рекламы'
    ]
    
    # Неоднозначные случаи (MAYBE)
    maybe_patterns = [
        r'\bэксклюзивно\b',  # может быть правдой
        r'\bсамый продаваемый\b',  # нужны источники
        r'\bлучший\b',  # может быть субъективным мнением
        r'\bофициальный поставщик\b',  # нужна проверка лицензии
        r'\bнаграждён медалью\b',  # нужна проверка
        r'\bоставьте контакт\b',  # ПДн - может быть легально
        r'\bзабронируйте\b.*\bконтакты\b',  # ПДн - может быть легально
        r'\bотзывы.*клиентов\b',  # может быть легально при согласии
        r'\bпродукт для диабетиков\b',  # может быть диетпродуктом
        r'\bдетское питание\b',  # нужна проверка контекста
    ]
    
    has_maybe_pattern = any(re.search(pattern, text_lower) for pattern in maybe_patterns)
    
    # Проверяем, есть ли НЕ-формальные нарушения
    has_non_formal = False
    
    # Если в описании есть жёсткие слова - это YES
    hard_keywords = [
        'запрещ', 'уголовн', 'пропаганд', 'незаконн',
        'лечит', 'излечива', 'гарантированн', 'наркотик'
    ]
    
    for keyword in hard_keywords:
        if keyword in description_lower:
            has_non_formal = True
            break
    
    if has_non_formal:
        return "yes"
    
    # Если есть паттерн MAYBE или только формальные нарушения - MAYBE
    if has_maybe_pattern:
        return "maybe"
    
    for keyword in formal_keywords:
        if keyword in description_lower:
            return "maybe"
    
    # По умолчанию для нарушений - MAYBE (безопасная сторона)
    return "maybe"

# ============================================================================
# ФОРМИРОВАНИЕ JSON (ТОЛЬКО 6 НАРУШЕНИЙ)
# ============================================================================

if result.get("violation_status") in ["Да", "Потенциальное нарушение"]:
    # Парсим все нарушения
    laws = result['law'].split(';')
    articles = result['article'].split(';')
    descriptions = result['description'].split(';')
    risks = result['risk'].split(';')
    recommendations = result['recommendation'].split(';')
    
    # Берем только первые 6
    laws = laws[:6]
    articles = articles[:6]
    descriptions = descriptions[:6]
    risks = risks[:6]
    recommendations = recommendations[:6]
    
    # УМНАЯ КЛАССИФИКАЦИЯ
    violation = decide_violation(my_text, result)
    
    output_json = {
        "law_name": "; ".join([l.strip() for l in laws]),
        "article_number": "; ".join([a.strip() for a in articles]),
        "description": "; ".join([d.strip() for d in descriptions]),
        "risk": "; ".join([r.strip() for r in risks]),
        "recomendation": "; ".join([rec.strip() for rec in recommendations]),
        "violation": violation,
        "token-count": 150
    }
    
else:
    output_json = {
        "law_name": "",
        "article_number": "",
        "description": "Нарушений не обнаружено",
        "risk": "",
        "recomendation": "",
        "violation": "no",
        "token-count": 100
    }

# ============================================================================
# 🤖 GIGA-CHECK: ФИНАЛЬНАЯ ПРОВЕРКА ЧЕРЕЗ GIGACHAT (С ДЕТАЛЬНЫМИ ПРАВИЛАМИ)
# ============================================================================

print("\n" + "="*80)
print("🤖 GIGA-CHECK: GigaChat проверяет логику отчёта...")
print("="*80)

giga_check_prompt = f"""Ты — юридический эксперт по российскому рекламному праву.

📝 ТЕКСТ РЕКЛАМЫ:
{my_text}

📊 МОЙ ОТЧЁТ:
{json.dumps(output_json, ensure_ascii=False, indent=2)}

🎯 ДЕТАЛЬНЫЕ ПРАВИЛА ДЛЯ КЛАССИФИКАЦИИ (YES / MAYBE / NO):

🔴 YES (жёсткие запреты и явные нарушения):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Наркотики, оружие, экстремизм/ненависть (УК РФ, ФЗ-3)
2. БАД/продукт "лечит", "вылечит", "излечивает" (ФЗ-61, ФЗ-38 ст.25)
3. "Гарантируем доход", "100% прибыль", "вложи-получи" (ФЗ-38 ст.5, ст.28)
4. Табак, порнография, непристойные слова (ФЗ-38 ст.20, ст.5)
5. ЯВНАЯ недостоверность:
   • "Продано более 1 млн" (реально 15к) → YES
   • "Мгновенная доставка" (факт: 2 дня) → YES
   • "Цена снижена на 80%" (завышена) → YES
   • "Единственная школа с дипломом" → YES
6. Порочащие сравнения: "их сервис — издевательство", "остальные — подделки" → YES
7. Призыв к незаконным действиям: "отомстить", "найдём компромат" → YES
8. Детская аудитория + лечебные обещания: "Детский чай лечит" → YES
9. Ложные официальные статусы: "Одобрено ВОЗ" (без док-в) → YES
10. Медфункции у не-медизделий: "Массажёр = МРТ" → YES
11. Ввод в заблуждение: "Обмен без вопросов", "Выиграй без условий" → YES

🟡 MAYBE (формальные требования, неоднозначные случаи):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Нет маркировки "Реклама" (может быть на креативе/лендинге) → MAYBE
2. Нет ИНН/ERID (может быть на баннере) → MAYBE
3. Условия акции не раскрыты (могут быть на сайте) → MAYBE
4. Сбор ПДн: "Оставьте контакты" (может быть легально с политикой) → MAYBE
5. Сомнительные утверждения БЕЗ ЯВНЫХ ДОКАЗАТЕЛЬСТВ ложности:
   • "Эксклюзивно" (может быть договор) → MAYBE
   • "Самый продаваемый" (нужны источники) → MAYBE
   • "Официальный поставщик Олимпиады" (нужна лицензия) → MAYBE
   • "Награждён медалью" (нужна проверка) → MAYBE
6. Отзывы с ФИО клиентов (может быть согласие) → MAYBE
7. "Продукт для диабетиков" (может быть диетпродукт) → MAYBE

✅ NO (нет нарушений):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Нейтральные товары/услуги без триггеров
• Корректная маркировка: "Реклама. ООО..., ИНН..., ERID..."
• Нет недостоверности, запретов, детской аудитории, медобещаний, гарантий дохода

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎯 ТВОЯ ЗАДАЧА:
1. Проверь текст рекламы на соответствие ДЕТАЛЬНЫМ ПРАВИЛАМ выше
2. Определи правильный статус: YES / MAYBE / NO
3. Если моя классификация НЕВЕРНАЯ - исправь поле "violation"

4. Верни ТОЛЬКО JSON:
{{
    "is_correct": true/false,
    "corrected_violation": "yes" или "maybe" или "no",
    "explanation": "краткое объяснение со ссылкой на правило"
}}

ВЕРНИ ТОЛЬКО JSON, БЕЗ КОММЕНТАРИЕВ!
"""

try:
    giga_response = client.chat.completions.create(
        model="GigaChat/GigaChat-2-Max-without-filter",
        messages=[
            {"role": "system", "content": "Ты — юридический эксперт по российскому рекламному праву. Ты знаешь все детальные правила классификации нарушений."},
            {"role": "user", "content": giga_check_prompt}
        ],
        temperature=0.3,
        max_tokens=500
    )
    
    giga_check_text = giga_response.choices[0].message.content.strip()
    
    # Парсим ответ GigaChat
    if giga_check_text.startswith("```"):
        giga_check_text = giga_check_text.split("```")[1]
        if giga_check_text.startswith("json"):
            giga_check_text = giga_check_text[4:]
    
    giga_check = json.loads(giga_check_text)
    
    print(f"\n✅ GigaChat проверил отчёт!")
    print(f"   • Правильно: {giga_check.get('is_correct')}")
    print(f"   • Объяснение: {giga_check.get('explanation')}")
    
    # Если GigaChat исправил классификацию - применяем
    if not giga_check.get('is_correct') and giga_check.get('corrected_violation'):
        old_violation = output_json['violation']
        output_json['violation'] = giga_check['corrected_violation']
        print(f"\n🔧 GigaChat исправил: {old_violation} → {giga_check['corrected_violation']}")
    
except Exception as e:
    print(f"\n⚠️  GigaChat не смог проверить (ошибка: {e})")
    print(f"   Использую локальную классификацию: {output_json['violation']}")

# ============================================================================
# ФИНАЛЬНЫЙ ВЫВОД
# ============================================================================

print("\n" + "="*80)
print("📊 ФИНАЛЬНЫЙ РЕЗУЛЬТАТ:")
print("="*80 + "\n")
print(json.dumps(output_json, ensure_ascii=False, indent=4))


AttributeError: 'GigaChatFineTuningAnalyzer' object has no attribute 'analyze'